# NYC Taxi Data Analysis (Optional)

This notebook shows how to use real NYC taxi data for traffic flow prediction.

**Note**: This is optional and requires downloading ~100MB of data. The main project uses SUMO for controllable data.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## Download Data (Run Once)

```bash
# Download January 2023 Yellow Taxi data
wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
```

In [ ]:
data_path = Path("../data/raw/yellow_tripdata_2023-01.parquet")
if data_path.exists():
    df = pd.read_parquet(data_path)
    print(f"Shape: {df.shape}")
    print(df.head())
else:
    print("Data not found. Download from NYC TLC website.")

## Feature Engineering

In [ ]:
if data_path.exists():
    df = df.copy()
    df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
    df["pickup_dayofweek"] = df["tpep_pickup_datetime"].dt.dayofweek
    df["trip_duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    df["speed_mph"] = df["trip_distance"] / (df["trip_duration_min"] / 60)
    
    # Filter reasonable trips
    df = df[
        (df["trip_distance"] > 0.1) &
        (df["trip_distance"] < 50) &
        (df["trip_duration_min"] > 1) &
        (df["trip_duration_min"] < 120) &
        (df["speed_mph"] > 1) &
        (df["speed_mph"] < 60)
    ]
    
    # Aggregate by zone pair and 15-min interval
    df["interval"] = df["tpep_pickup_datetime"].dt.floor("15min")
    zone_pairs = df.groupby(["interval", "PULocationID", "DOLocationID"]).agg(
        trip_count=("VendorID", "count"),
        avg_speed=("speed_mph", "mean"),
        avg_distance=("trip_distance", "mean"),
    ).reset_index()
    
    print(f"Zone pairs: {zone_pairs[['PULocationID', 'DOLocationID']].drop_duplicates().shape[0]}")
    print(f"Time intervals: {zone_pairs['interval'].nunique()}")

## Train Model

In [ ]:
if data_path.exists():
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error
    import joblib
    
    # Prepare features
    zone_pairs["hour"] = zone_pairs["interval"].dt.hour
    zone_pairs["dayofweek"] = zone_pairs["interval"].dt.dayofweek
    zone_pairs["zone_pair"] = zone_pairs["PULocationID"].astype(str) + "_" + zone_pairs["DOLocationID"].astype(str)
    
    # One-hot encode zone pairs (top 50)
    top_pairs = zone_pairs["zone_pair"].value_counts().head(50).index
    zone_pairs = zone_pairs[zone_pairs["zone_pair"].isin(top_pairs)]
    
    features = pd.get_dummies(zone_pairs["zone_pair"], prefix="zone")
    features["hour"] = zone_pairs["hour"]
    features["dayofweek"] = zone_pairs["dayofweek"]
    
    target = zone_pairs["trip_count"]
    
    # Time-based split
    split_idx = int(len(zone_pairs) * 0.8)
    X_train, X_test = features[:split_idx], features[split_idx:]
    y_train, y_test = target[:split_idx], target[split_idx:]
    
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print(f"MAE: {mae:.2f}")
    
    # Save
    joblib.dump({
        "model": model,
        "features": list(features.columns),
        "target": "trip_count",
        "metrics": {"mae": mae},
        "model_type": "random_forest",
        "version": "1.0",
    }, "../models/nyc_taxi_rf.joblib")
    print("Model saved!")